# 🌾 DehatiAI — ResNet-50 + CBAM Crop Disease Training
### ⚡ Ultra-Fast Cloud GPU Training (Nvidia T4 / A100)

This notebook trains the production **ResNet-50 + CBAM (Convolutional Block Attention Module)** architecture on **32+ agricultural crop disease classes** (including Pakistani wheat rusts, potato blights, and tomato viruses).

---
**Architectural Highlights:**
- **CBAM Attention:** Focuses on pathological lesions while ignoring field soil, mud, and background foliage.
- **Field Augmentation:** Simulates motion blur (`GaussianBlur`), harsh sunlight (`ColorJitter`), and leaf occlusion (`RandomErasing`).
- **Auto-Export:** Exports directly to production-ready `resnet50_cbam.onnx` (<50ms inference, <60MB RAM).

In [ ]:
# Step 1: Check GPU Acceleration & Install Dependencies
!nvidia-smi
!pip install -q onnx onnxruntime tqdm matplotlib pillow

In [ ]:
# Step 2: Download & Prepare Combined Multi-Class Field Dataset
import os, shutil, zipfile, glob, re

print('Downloading PlantDoc Dataset (28 field classes)...')
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git

print('Downloading Roboflow Wheat Dataset (Black, Brown, Yellow Rust, Healthy Wheat)...')
!curl -L 'https://universe.roboflow.com/ds/tZHc1dmGFO?key=Ab6rjyPIZS' -o roboflow.zip
!unzip -q roboflow.zip -d roboflow_wheat && rm roboflow.zip

# Organize into PyTorch ImageFolder format
DATA_DIR = './data/crop_disease'
os.makedirs(f'{DATA_DIR}/train', exist_ok=True)
os.makedirs(f'{DATA_DIR}/val', exist_ok=True)

# 1. Copy PlantDoc classes
def clean_name(n):
    return re.sub(r'[^a-z0-9]+', '_', n.lower().replace(' leaf', '')).strip('_')

for split in ['train', 'test']:
    src_split = f'PlantDoc-Dataset/{split}'
    dst_split = 'val' if split == 'test' else 'train'
    if os.path.exists(src_split):
        for cls in os.listdir(src_split):
            cls_clean = clean_name(cls)
            src_cls = os.path.join(src_split, cls)
            dst_cls = os.path.join(DATA_DIR, dst_split, cls_clean)
            os.makedirs(dst_cls, exist_ok=True)
            for img in os.listdir(src_cls):
                if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                    clean_img_name = re.sub(r'[\<\>\:\"\/\\\|\?\*\&\%]', '_', img)
                    try:
                        shutil.copy2(os.path.join(src_cls, img), os.path.join(dst_cls, clean_img_name))
                    except Exception:
                        pass

# 2. Copy Roboflow Wheat classes
WHEAT_MAP = {
    'Black-Rust': 'wheat_black_rust',
    'Brown-Rust': 'wheat_brown_rust',
    'Healthy-Wheat': 'wheat_healthy',
    'Yellow-Rust': 'wheat_yellow_rust'
}
for split in ['train', 'valid', 'test']:
    dst_split = 'train' if split == 'train' else 'val'
    img_dir = f'roboflow_wheat/{split}/images'
    if os.path.exists(img_dir):
        for img in os.listdir(img_dir):
            prefix = img.split('_')[0]
            if prefix in WHEAT_MAP:
                target_cls = WHEAT_MAP[prefix]
                dst_cls = os.path.join(DATA_DIR, dst_split, target_cls)
                os.makedirs(dst_cls, exist_ok=True)
                shutil.copy2(os.path.join(img_dir, img), os.path.join(dst_cls, f'rf_{img}'))

train_classes = sorted(os.listdir(f'{DATA_DIR}/train'))
print(f'\nDataset Ready! Total Classes: {len(train_classes)}')


In [ ]:
# Step 3: ResNet-50 + CBAM Neural Network Architecture
import torch
import torch.nn as nn
from torchvision import models

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        reduced = max(in_channels // reduction_ratio, 8)
        self.shared_mlp = nn.Sequential(
            nn.Linear(in_channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, in_channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_pool = torch.mean(x, dim=(2, 3))
        max_pool = torch.amax(x, dim=(2, 3))
        avg_out = self.shared_mlp(avg_pool)
        max_out = self.shared_mlp(max_pool)
        att = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * att

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_pool = torch.mean(x, dim=1, keepdim=True)
        max_pool = torch.amax(x, dim=1, keepdim=True)
        concat = torch.cat([avg_pool, max_pool], dim=1)
        att = self.sigmoid(self.conv(concat))
        return x * att

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_channels, reduction_ratio)
        self.sa = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.sa(self.ca(x))

class ResNet50_CBAM(nn.Module):
    def __init__(self, num_classes=32, pretrained=True, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.cbam = CBAM(in_channels=2048, reduction_ratio=16, kernel_size=7)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(512, num_classes)
        )
        # Freeze early layers, fine-tune last 2 blocks + CBAM + classifier
        for p in self.parameters():
            p.requires_grad = False
        for p in self.cbam.parameters():
            p.requires_grad = True
        for p in self.classifier.parameters():
            p.requires_grad = True
        for b in list(self.layer4.children())[-2:]:
            for p in b.parameters():
                p.requires_grad = True

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.cbam(x)
        x = torch.flatten(self.avgpool(x), 1)
        return self.classifier(x)

print('Architecture defined with CBAM Attention!')

In [ ]:
# Step 4: Pakistani Field Data Augmentation & DataLoader
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.3),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.8)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.12)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transforms)
val_ds   = datasets.ImageFolder(f'{DATA_DIR}/val', transform=val_transforms)

targets = [s[1] for s in train_ds.samples]
counts = torch.clamp(torch.bincount(torch.tensor(targets)), min=1)
weights = 1.0 / counts.float()
sampler = WeightedRandomSampler(weights[targets], num_samples=len(targets), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples: {len(train_ds)} | Val samples: {len(val_ds)} | Classes: {len(train_ds.classes)}')

In [ ]:
# Step 5: GPU Accelerated Training Loop (Fast 25 Epochs with CUDA)
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

num_classes = len(train_ds.classes)
model = ResNet50_CBAM(num_classes=num_classes, pretrained=True).to(device)

cw = (1.0 / counts.float())
cw = (cw / cw.sum() * len(counts)).to(device)
criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

optimizer = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.cbam.parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)

scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

EPOCHS = 25
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    r_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{EPOCHS} [Train]')
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        r_loss += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'loss': f'{r_loss/total:.4f}', 'acc': f'{100.*correct/total:.2f}%'})

    # Validation
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            _, pred = out.max(1)
            v_correct += pred.eq(labels).sum().item()
            v_total += labels.size(0)
    v_acc = 100.0 * v_correct / max(v_total, 1)
    t_acc = 100.0 * correct / max(total, 1)
    scheduler.step()

    print(f'\nEpoch {epoch:02d} Summary | Train Acc: {t_acc:.2f}% | Val Acc: {v_acc:.2f}%')
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'val_acc': v_acc,
            'classes': train_ds.classes
        }, 'best_model.pth')
        print(f'   Saved best model! (Val Acc: {v_acc:.2f}%)')

print(f'\nTraining Complete! Best Validation Accuracy: {best_val_acc:.2f}%')

In [ ]:
# Step 6: Export to ONNX (<50ms Node.js inference) & Auto-Download
print('Exporting to ONNX format...')
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(
    model,
    dummy_input,
    'resnet50_cbam.onnx',
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    opset_version=14
)
print('Successfully exported resnet50_cbam.onnx!')

import json
with open('class_names.json', 'w') as f:
    json.dump(train_ds.classes, f, indent=2)
print('Saved class_names.json!')

try:
    from google.colab import files
    print('Downloading resnet50_cbam.onnx and class_names.json to your computer...')
    files.download('resnet50_cbam.onnx')
    files.download('class_names.json')
except Exception:
    print('Download ready.')